# 🎬 AutoClip: สตูดิโอหนังสั้น AI อัตโนมัติ
กด **Runtime → Run all** (หรือกด ▶ ทีละช่อง 1→4) แล้วรอลิงก์เว็บในช่อง 4

ครั้งแรกต้องตั้ง **Secrets (🔑 แถบซ้าย)** ก่อน: `OPENROUTER_API_KEY` (จำเป็น) และ `AUTOCLIP_PASSWORD` (แนะนำ) แล้วเปิดสวิตช์ Notebook access ทั้งคู่

โค้ดดึงจาก GitHub อัตโนมัติ (อัปเดตล่าสุดทุกครั้งที่รัน) · ข้อมูลทั้งหมด (สกิล/โปรเจกต์/คลิป) อยู่ใน Google Drive → `My Drive/AutoClip`

In [ ]:
#@title 1) ดึงโค้ดจาก GitHub + เชื่อม Google Drive
import os, subprocess, shutil
REPO = 'https://github.com/naretchartham2525-cpu/web-animation'
BRANCH = 'main'  #@param {type:"string"}
CODE = '/content/web-animation'

if os.path.exists(f'{CODE}/.git'):
    r = subprocess.run(f'git -C {CODE} fetch -q --depth 1 origin {BRANCH} && git -C {CODE} reset -q --hard origin/{BRANCH}',
                       shell=True, capture_output=True, text=True)
else:
    r = subprocess.run(f'git clone -q --depth 1 -b {BRANCH} {REPO} {CODE}', shell=True, capture_output=True, text=True)
if r.returncode != 0:
    raise SystemExit('❌ ดึงโค้ดไม่สำเร็จ (repo ต้องเป็น public): ' + r.stderr)
print('✅ โค้ด:', subprocess.run(f'git -C {CODE} log -1 --format="%h %s (%cr)"', shell=True, capture_output=True, text=True).stdout.strip())

from google.colab import drive
drive.mount('/content/drive')
ROOT = '/content/drive/MyDrive/AutoClip'
for d in ('skills', 'projects', 'fonts'):
    os.makedirs(f'{ROOT}/{d}', exist_ok=True)
if not os.listdir(f'{ROOT}/skills'):   # ครั้งแรก: ใส่สกิลตัวอย่างให้
    shutil.copytree(f'{CODE}/skills', f'{ROOT}/skills', dirs_exist_ok=True)
print('✅ Drive:', ROOT)
print('   สกิล:', sorted(os.listdir(f'{ROOT}/skills')))
print('   โปรเจกต์:', sorted(os.listdir(f'{ROOT}/projects')))

In [ ]:
#@title 2) ติดตั้งเครื่องมือ (ฟอนต์ไทย, เสียงฟรี, ตัดคำไทย) ~1 นาที
import subprocess
subprocess.run('apt-get -qq update > /dev/null; apt-get -qq install -y fonts-thai-tlwg > /dev/null', shell=True)
subprocess.run('pip -q install flask edge-tts pythainlp', shell=True)
subprocess.run('fc-cache -f > /dev/null', shell=True)
print(subprocess.run('ffmpeg -version', shell=True, capture_output=True, text=True).stdout.splitlines()[0])
print('✅ ติดตั้งเสร็จ')

In [ ]:
#@title 3) โหลด API key จาก Secrets (ไม่โชว์ key)
import os, secrets as _s
from google.colab import userdata

def _get(name):
    try:
        v = userdata.get(name)
        return v.strip() if v else ''
    except Exception:
        return ''

key = _get('OPENROUTER_API_KEY')
if key:
    os.environ['OPENROUTER_API_KEY'] = key
    print('✅ OPENROUTER_API_KEY: โหลดแล้ว (' + key[:6] + '…)')
else:
    print('❌ ไม่พบ OPENROUTER_API_KEY → แถบซ้าย 🔑 Secrets → Add new secret → เปิด Notebook access')

pw = _get('AUTOCLIP_PASSWORD')
if not pw:
    pw = _s.token_urlsafe(8)
    print('⚠️ ไม่ได้ตั้ง AUTOCLIP_PASSWORD → สุ่มรหัสให้ (เปลี่ยนทุกครั้งที่รัน):', pw)
else:
    print('✅ AUTOCLIP_PASSWORD: โหลดแล้ว')
os.environ['AUTOCLIP_PASSWORD'] = pw

In [ ]:
#@title 4) เปิดเว็บ + Cloudflare Tunnel 🌐
import subprocess, time, re, os, requests

os.environ['AUTOCLIP_ROOT'] = ROOT
os.environ['PYTHONDONTWRITEBYTECODE'] = '1'
os.environ['PYTHONUNBUFFERED'] = '1'
subprocess.run('pkill -f "app.server"; pkill -f cloudflared', shell=True)
time.sleep(1)

server = subprocess.Popen(['python', '-m', 'app.server', '--port', '8000'], cwd=CODE,
                          stdout=open('/content/server.log', 'w'), stderr=subprocess.STDOUT, env=os.environ.copy())
for _ in range(40):
    try:
        requests.get('http://127.0.0.1:8000/api/ping', timeout=1); break
    except Exception:
        time.sleep(1)
else:
    print(open('/content/server.log').read()[-3000:])
    raise SystemExit('❌ เซิร์ฟเวอร์ไม่ขึ้น ดู log ด้านบน')

CF = '/content/cloudflared'
if not os.path.exists(CF):
    subprocess.run(f'wget -q -O {CF} https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 && chmod +x {CF}', shell=True)
tunnel = subprocess.Popen([CF, 'tunnel', '--no-autoupdate', '--url', 'http://127.0.0.1:8000'],
                          stdout=open('/content/cf.log', 'w'), stderr=subprocess.STDOUT)
url = None
for _ in range(60):
    m = re.search(r'https://[a-z0-9-]+\.trycloudflare\.com', open('/content/cf.log').read())
    if m:
        url = m.group(0); break
    time.sleep(1)

print('=' * 60)
if url:
    print('🌐 เปิดเว็บที่:', url)
    print('🔒 รหัสผ่าน: ใช้ AUTOCLIP_PASSWORD (หรือรหัสที่สุ่มในช่อง 3)')
    print('⏳ ถ้าเปิดแล้วขึ้น error ให้รอ 10-20 วินาทีแล้วรีเฟรช')
else:
    print('❌ ไม่ได้ URL'); print(open('/content/cf.log').read()[-2000:])
print('=' * 60)
print('⚠️ ห้ามปิดแท็บ Colab นี้ ถ้าปิด เว็บจะหยุด (งานที่ทำแล้วอยู่ใน Drive ไม่หาย)')

In [ ]:
#@title 5) (ไม่บังคับ) ดู log เซิร์ฟเวอร์ ล่าสุด
print(open('/content/server.log').read()[-5000:])